# Lifecycle-Stratified Supervised Cluster Labeling: LightGBM & XGBoost

This notebook trains supervised models to reproduce the existing V11 lifecycle cluster assignments using the same stratum-specific feature lists used in the HDBSCAN pipeline.

Important framing: LightGBM and XGBoost are supervised classifiers, not unsupervised clustering algorithms. In this notebook, the existing V11 `cluster_key` labels are treated as the target. The supervised models learn a scalable mapping from developer features to cluster labels. This is useful for:

- validating whether clusters are explainable from the selected features
- assigning cluster labels to new/future developers without rerunning HDBSCAN
- comparing LightGBM vs XGBoost as supervised cluster-label surrogates
- saving a separate set of cluster-assignment tables without overwriting V11 HDBSCAN outputs

Primary saved DuckDB tables:

- `dev_supervised_cluster_membership_v1_lgbm`
- `dev_supervised_cluster_membership_v1_xgb`
- `dev_supervised_cluster_membership_v1_final`
- `dev_supervised_cluster_run_stats_v1`
- `dev_supervised_cluster_profile_summary_v1`


## 0. Setup

In [12]:
import gc
import json
import warnings
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DB_PATH = "developer_project.duckdb"
PROFILE_TABLE = "dev_profile_final_v4"
SOURCE_CLUSTER_TABLE = "dev_lifecycle_cluster_membership_v11_final"
ID_COL = "developer_id"

OUTPUT_TABLE_PREFIX = "dev_supervised_cluster_membership_v1"
LGBM_TABLE = f"{OUTPUT_TABLE_PREFIX}_lgbm"
XGB_TABLE = f"{OUTPUT_TABLE_PREFIX}_xgb"
FINAL_TABLE = f"{OUTPUT_TABLE_PREFIX}_final"
RUN_STATS_TABLE = "dev_supervised_cluster_run_stats_v1"
PROFILE_SUMMARY_TABLE = "dev_supervised_cluster_profile_summary_v1"

ARTIFACT_DIR = Path("supervised_cluster_artifacts_v1")
ARTIFACT_DIR.mkdir(exist_ok=True)

# Keeps first run practical. Increase if runtime is acceptable.
MAX_TRAIN_ROWS_PER_STRATUM = 250_000
TEST_SIZE = 0.20

# Whether to train noise clusters as explicit classes.
# Set False if you want supervised models to learn only meaningful non-noise clusters.
INCLUDE_NOISE_AS_CLASS = True

con = duckdb.connect(DB_PATH)
print("Connected to:", DB_PATH)


Connected to: developer_project.duckdb


## 1. Lifecycle-first feature configuration

In [2]:
FEATURES_BY_STRATUM = {
    "active": [
        "log_activity_count_0_30d",
        "log_activity_count_30_90d",
        "unique_activity_types_0_30d",
        "unique_modalities_0_30d",
        "developer_effort_score",
        "weighted_recent_confidence_effort",
        "recent_build_flag",
        "log_build_count_0_30d",
        "build_share_lifetime",
        "activity_velocity_0_30_vs_30_90",
        "log_clipped_lifetime_activity_count_p99",
        "persona_entropy",
    ],
    "cooling": [
        "log_activity_count_30_90d",
        "log_activity_count_90_180d",
        "developer_effort_score",
        "weighted_recent_confidence_effort",
        "build_share_lifetime",
        "high_effort_share_lifetime",
        "log_clipped_lifetime_activity_count_p99",
        "persona_entropy",
    ],
    "at_risk": [
        "log_activity_count_30_90d",
        "log_activity_count_90_180d",
        "developer_effort_score",
        "build_share_lifetime",
        "high_effort_share_lifetime",
        "log_clipped_lifetime_activity_count_p99",
        "persona_entropy",
    ],
    # Used for lightweight dormant carry-forward only, not supervised model fitting.
    "dormant": [
        "developer_effort_score",
        "build_share_lifetime",
        "high_effort_share_lifetime",
        "log_clipped_lifetime_activity_count_p99",
        "persona_entropy",
    ],
}

INTERPRETATION_FEATURES = [
    "avg_effort_rank_0_30d",
    "avg_effort_rank_30_90d",
    "avg_effort_rank_90_180d",
    "has_high_effort_0_30d",
    "active_non_builder_0_30d",
    "low_volume_builder_0_30d",
    "has_activity_0_30d",
    "has_activity_30_90d",
    "has_activity_90_180d",
    "log_build_count_30_90d",
    "log_build_count_90_180d",
]

VALID_STRATA = ["active", "cooling", "at_risk", "dormant", "unactivated"]
MODEL_STRATA = ["active", "cooling", "at_risk"]
LIGHTWEIGHT_STRATA = ["dormant"]
PSEUDO_STRATA = ["unactivated", "unknown"]

ALL_MODEL_FEATURES = sorted({f for s in MODEL_STRATA for f in FEATURES_BY_STRATUM[s]})
ALL_LIGHTWEIGHT_FEATURES = sorted({f for s in LIGHTWEIGHT_STRATA for f in FEATURES_BY_STRATUM[s]})
ALL_FEATURES = sorted(set(ALL_MODEL_FEATURES) | set(ALL_LIGHTWEIGHT_FEATURES) | set(INTERPRETATION_FEATURES))

print("Model strata:", MODEL_STRATA)
print("Total unique features loaded:", len(ALL_FEATURES))
for stratum in MODEL_STRATA:
    print(f"{stratum}: {len(FEATURES_BY_STRATUM[stratum])} supervised features")


Model strata: ['active', 'cooling', 'at_risk']
Total unique features loaded: 25
active: 12 supervised features
cooling: 8 supervised features
at_risk: 7 supervised features


## 2. Check source tables and model availability

In [3]:
def table_exists(con, table_name):
    return con.execute(f"""
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_name = '{table_name}'
    """).fetchone()[0] > 0

required = [PROFILE_TABLE, SOURCE_CLUSTER_TABLE]
missing = [t for t in required if not table_exists(con, t)]
if missing:
    raise ValueError(f"Missing required DuckDB tables: {missing}. Run the V11 HDBSCAN clustering notebook first.")

profile_cols = con.execute(f"DESCRIBE {PROFILE_TABLE}").df()["column_name"].tolist()
source_cols = con.execute(f"DESCRIBE {SOURCE_CLUSTER_TABLE}").df()["column_name"].tolist()

required_source_cols = [ID_COL, "stratum", "cluster_key"]
missing_source_cols = [c for c in required_source_cols if c not in source_cols]
if missing_source_cols:
    raise ValueError(f"Missing required columns from {SOURCE_CLUSTER_TABLE}: {missing_source_cols}")

existing_features = [c for c in ALL_FEATURES if c in profile_cols]
missing_features = [c for c in ALL_FEATURES if c not in profile_cols]
print("Existing requested features:", len(existing_features))
if missing_features:
    print("Missing features skipped:")
    for c in missing_features:
        print("-", c)

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception as exc:
    HAS_LGBM = False
    print("LightGBM is not available. Install with: pip install lightgbm")

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception as exc:
    HAS_XGB = False
    print("XGBoost is not available. Install with: pip install xgboost")

if not HAS_LGBM and not HAS_XGB:
    raise ImportError("Neither LightGBM nor XGBoost is installed. Install at least one before running this notebook.")

print("HAS_LGBM:", HAS_LGBM)
print("HAS_XGB:", HAS_XGB)


Existing requested features: 25
HAS_LGBM: True
HAS_XGB: True


## 3. Helper functions

In [4]:
def load_stratum_training_frame(stratum, features):
    cols = [ID_COL] + [c for c in features if c in profile_cols]
    select_cols = ", ".join([f"p.{c}" for c in cols])
    noise_filter = "" if INCLUDE_NOISE_AS_CLASS else "AND LOWER(CAST(m.cluster_key AS VARCHAR)) NOT LIKE '%noise%'"
    sql = f"""
        SELECT
            {select_cols},
            m.stratum,
            CAST(m.cluster_key AS VARCHAR) AS source_cluster_key
        FROM {SOURCE_CLUSTER_TABLE} m
        JOIN {PROFILE_TABLE} p USING ({ID_COL})
        WHERE LOWER(CAST(m.stratum AS VARCHAR)) = '{stratum}'
          {noise_filter}
    """
    return con.execute(sql).df()


def stratified_train_sample(df, label_col="source_cluster_key", max_rows=MAX_TRAIN_ROWS_PER_STRATUM):
    if len(df) <= max_rows:
        return df.copy()
    counts = df[label_col].value_counts()
    target = max_rows
    pieces = []
    for label, n in counts.items():
        frac = n / len(df)
        take = max(100, int(round(frac * target)))
        take = min(take, n)
        pieces.append(df[df[label_col] == label].sample(n=take, random_state=RANDOM_STATE))
    out = pd.concat(pieces, ignore_index=True)
    if len(out) > max_rows:
        out = out.sample(n=max_rows, random_state=RANDOM_STATE).reset_index(drop=True)
    return out


def make_preprocessor():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])


def make_lgbm(n_classes):
    return LGBMClassifier(
        objective="multiclass" if n_classes > 2 else "binary",
        n_estimators=400,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=64,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )


def make_xgb(n_classes):
    return XGBClassifier(
        objective="multi:softprob" if n_classes > 2 else "binary:logistic",
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric="mlogloss" if n_classes > 2 else "logloss",
        tree_method="hist",
    )


def predict_full_stratum(stratum, features, model_bundle, model_name):
    feature_cols = model_bundle["features"]
    cols = [ID_COL] + feature_cols
    select_cols = ", ".join([f"p.{c}" for c in cols])
    sql = f"""
        SELECT {select_cols}
        FROM {SOURCE_CLUSTER_TABLE} m
        JOIN {PROFILE_TABLE} p USING ({ID_COL})
        WHERE LOWER(CAST(m.stratum AS VARCHAR)) = '{stratum}'
    """
    full_df = con.execute(sql).df()
    X = full_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    Xt = model_bundle["preprocessor"].transform(X)
    pred_int = model_bundle["model"].predict(Xt)
    pred_label = model_bundle["label_encoder"].inverse_transform(pred_int.astype(int))
    proba = model_bundle["model"].predict_proba(Xt)
    max_proba = proba.max(axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        entropy = -np.sum(np.where(proba > 0, proba * np.log(proba), 0.0), axis=1)
    out = pd.DataFrame({
        ID_COL: full_df[ID_COL].values,
        "stratum": stratum,
        "supervised_model": model_name,
        "predicted_cluster_key": pred_label,
        "predicted_cluster_id": pred_int.astype("int32"),
        "prediction_confidence": max_proba.astype("float32"),
        "prediction_entropy": entropy.astype("float32"),
    })
    return out


def save_df_to_table(con, df, table_name):
    con.register("tmp_save_df", df)
    con.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM tmp_save_df")
    con.unregister("tmp_save_df")


def append_df_to_table(con, df, table_name):
    con.register("tmp_append_df", df)
    con.execute(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM tmp_append_df WHERE 1 = 0")
    con.execute(f"INSERT INTO {table_name} SELECT * FROM tmp_append_df")
    con.unregister("tmp_append_df")


## 4. Train LightGBM and XGBoost per modeled stratum

In [5]:
model_bundles = {"lgbm": {}, "xgb": {}}
run_rows = []

# Reset run stats table for this supervised run.
con.execute(f"DROP TABLE IF EXISTS {RUN_STATS_TABLE}")

for stratum in MODEL_STRATA:
    print("\n" + "=" * 80)
    print(f"Training supervised cluster labelers for stratum: {stratum}")
    print("=" * 80)

    features = [f for f in FEATURES_BY_STRATUM[stratum] if f in profile_cols]
    df = load_stratum_training_frame(stratum, features)
    print("Full labeled rows:", len(df))
    print("Source cluster distribution:")
    display(df["source_cluster_key"].value_counts().reset_index(name="n"))

    train_df = stratified_train_sample(df)
    print("Training sample rows:", len(train_df))

    le = LabelEncoder()
    y = le.fit_transform(train_df["source_cluster_key"].astype(str))
    n_classes = len(le.classes_)
    print("Classes:", list(le.classes_))

    X = train_df[features].replace([np.inf, -np.inf], np.nan)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y if n_classes > 1 else None,
    )

    pre = make_preprocessor()
    X_train_t = pre.fit_transform(X_train)
    X_test_t = pre.transform(X_test)

    candidates = []
    if HAS_LGBM:
        candidates.append(("lgbm", make_lgbm(n_classes)))
    if HAS_XGB:
        candidates.append(("xgb", make_xgb(n_classes)))

    for model_name, model in candidates:
        print(f"\nFitting {model_name.upper()} for {stratum}...")
        model.fit(X_train_t, y_train)
        pred = model.predict(X_test_t)
        acc = accuracy_score(y_test, pred)
        bal_acc = balanced_accuracy_score(y_test, pred)
        macro_f1 = f1_score(y_test, pred, average="macro")
        weighted_f1 = f1_score(y_test, pred, average="weighted")
        print(f"{model_name} | accuracy={acc:.4f} balanced_accuracy={bal_acc:.4f} macro_f1={macro_f1:.4f} weighted_f1={weighted_f1:.4f}")

        bundle = {
            "model": model,
            "preprocessor": pre,
            "label_encoder": le,
            "features": features,
            "stratum": stratum,
            "model_name": model_name,
        }
        model_bundles[model_name][stratum] = bundle
        joblib.dump(bundle, ARTIFACT_DIR / f"{model_name}_{stratum}_supervised_cluster_v1.joblib")

        run_rows.append({
            "stratum": stratum,
            "supervised_model": model_name,
            "n_full_labeled_rows": len(df),
            "n_train_sample_rows": len(train_df),
            "n_features": len(features),
            "features": ", ".join(features),
            "n_classes": n_classes,
            "classes": ", ".join(le.classes_),
            "accuracy": acc,
            "balanced_accuracy": bal_acc,
            "macro_f1": macro_f1,
            "weighted_f1": weighted_f1,
        })

    del df, train_df, X, X_train, X_test, X_train_t, X_test_t
    gc.collect()

run_stats_df = pd.DataFrame(run_rows)
save_df_to_table(con, run_stats_df, RUN_STATS_TABLE)
display(run_stats_df)
print(f"Saved run stats table: {RUN_STATS_TABLE}")



Training supervised cluster labelers for stratum: active
Full labeled rows: 418049
Source cluster distribution:


,source_cluster_key,n
0,active_5,155026
1,active_noise,105723
2,active_1,66683
3,active_3,29395
4,active_2,24653
5,active_4,18554
6,active_0,18015


Training sample rows: 250000
Classes: ['active_0', 'active_1', 'active_2', 'active_3', 'active_4', 'active_5', 'active_noise']

Fitting LGBM for active...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lgbm | accuracy=0.9987 balanced_accuracy=0.9984 macro_f1=0.9984 weighted_f1=0.9987

Fitting XGB for active...
xgb | accuracy=0.9985 balanced_accuracy=0.9982 macro_f1=0.9982 weighted_f1=0.9985

Training supervised cluster labelers for stratum: cooling
Full labeled rows: 356500
Source cluster distribution:


...

Training sample rows: 250000
Classes: ['at_risk_0', 'at_risk_1', 'at_risk_2', 'at_risk_3', 'at_risk_4', 'at_risk_5', 'at_risk_noise']

Fitting LGBM for at_risk...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lgbm | accuracy=0.9964 balanced_accuracy=0.9943 macro_f1=0.9948 weighted_f1=0.9964

Fitting XGB for at_risk...
xgb | accuracy=0.9960 balanced_accuracy=0.9930 macro_f1=0.9941 weighted_f1=0.9960


,stratum,supervised_model,n_full_labeled_rows,n_train_sample_rows,n_features,features,n_classes,classes,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,active,lgbm,418049,250000,12,"log_activity_count_0_30d, log_activity_count_3...",7,"active_0, active_1, active_2, active_3, active...",0.99872,0.998422,0.998397,0.998720
1,active,xgb,418049,250000,12,"log_activity_count_0_30d, log_activity_count_3...",7,"active_0, active_1, active_2, active_3, active...",0.99846,0.998216,0.998228,0.998460
2,cooling,lgbm,356500,250000,8,"log_activity_count_30_90d, log_activity_count_...",8,"cooling_0, cooling_1, cooling_2, cooling_3, co...",0.99740,0.997699,0.997325,0.997402
3,cooling,xgb,356500,250000,8,"log_activity_count_30_90d, log_activity_count_...",8,"cooling_0, cooling_1, cooling_2, cooling_3, co...",0.99638,0.996950,0.996354,0.996384
4,at_risk,lgbm,1580877,250000,7,"log_activity_count_30_90d, log_activity_count_...",7,"at_risk_0, at_risk_1, at_risk_2, at_risk_3, at...",0.99642,0.994331,0.994760,0.996414
5,at_risk,xgb,1580877,250000,7,"log_activity_count_30_90d, log_activity_count_...",7,"at_risk_0, at_risk_1, at_risk_2, at_risk_3, at...",0.99600,0.993011,0.994126,0.995981


Saved run stats table: dev_supervised_cluster_run_stats_v1


## 5. Predict full modeled strata and save model-specific tables

In [6]:
def build_model_specific_membership(model_name, table_name):
    rows = []
    for stratum in MODEL_STRATA:
        if stratum not in model_bundles.get(model_name, {}):
            print(f"Skipping {model_name}/{stratum}: no fitted model bundle")
            continue
        pred_df = predict_full_stratum(
            stratum=stratum,
            features=FEATURES_BY_STRATUM[stratum],
            model_bundle=model_bundles[model_name][stratum],
            model_name=model_name,
        )
        rows.append(pred_df)
        print(f"{model_name}/{stratum}: predicted {len(pred_df):,} rows")

    if not rows:
        print(f"No rows for {model_name}; table not created.")
        return None

    out = pd.concat(rows, ignore_index=True)

    # Carry forward dormant and pseudo strata from the source V11 membership table.
    carry_sql = f"""
        SELECT
            {ID_COL},
            CAST(stratum AS VARCHAR) AS stratum,
            '{model_name}' AS supervised_model,
            CAST(cluster_key AS VARCHAR) AS predicted_cluster_key,
            CAST(hdbscan_cluster AS INTEGER) AS predicted_cluster_id,
            CAST(1.0 AS FLOAT) AS prediction_confidence,
            CAST(0.0 AS FLOAT) AS prediction_entropy
        FROM {SOURCE_CLUSTER_TABLE}
        WHERE LOWER(CAST(stratum AS VARCHAR)) IN ('dormant', 'unactivated', 'unknown')
    """
    carry_df = con.execute(carry_sql).df()
    out = pd.concat([out, carry_df], ignore_index=True)

    save_df_to_table(con, out, table_name)
    print(f"Saved {table_name}: {len(out):,} rows")
    display(out.groupby(["stratum", "predicted_cluster_key"]).size().reset_index(name="n").head(30))
    return out

lgbm_out = build_model_specific_membership("lgbm", LGBM_TABLE) if HAS_LGBM else None
xgb_out = build_model_specific_membership("xgb", XGB_TABLE) if HAS_XGB else None


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lgbm/active: predicted 418,049 rows


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lgbm/cooling: predicted 356,500 rows


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lgbm/at_risk: predicted 1,580,877 rows
100% ▕██████████████████████████████████████▏ (00:00:02.64 elapsed)     
Saved dev_supervised_cluster_membership_v1_lgbm: 9,381,508 rows


,stratum,predicted_cluster_key,n
0,active,active_0,18015
1,active,active_1,66682
2,active,active_2,24658
3,active,active_3,29409
4,active,active_4,18549
5,active,active_5,155034
6,active,active_noise,105702
7,at_risk,at_risk_0,436825
8,at_risk,at_risk_1,145071
9,at_risk,at_risk_2,210376


xgb/active: predicted 418,049 rows
xgb/cooling: predicted 356,500 rows
xgb/at_risk: predicted 1,580,877 rows
100% ▕██████████████████████████████████████▏ (00:00:02.38 elapsed)     
Saved dev_supervised_cluster_membership_v1_xgb: 9,381,508 rows


,stratum,predicted_cluster_key,n
0,active,active_0,18011
1,active,active_1,66737
2,active,active_2,24656
3,active,active_3,29462
4,active,active_4,18548
5,active,active_5,155018
6,active,active_noise,105617
7,at_risk,at_risk_0,437192
8,at_risk,at_risk_1,145077
9,at_risk,at_risk_2,210376


## 6. Build final supervised cluster membership table

For each stratum, select the model with the highest validation `weighted_f1`. Dormant and pseudo-groups are carried forward from the selected model output.

In [7]:
if not table_exists(con, RUN_STATS_TABLE):
    raise ValueError("Run stats table missing. Run training cells first.")

stats = con.execute(f"SELECT * FROM {RUN_STATS_TABLE}").df()
best_by_stratum = (
    stats.sort_values(["stratum", "weighted_f1", "balanced_accuracy"], ascending=[True, False, False])
    .groupby("stratum", as_index=False)
    .head(1)
    [["stratum", "supervised_model", "weighted_f1", "balanced_accuracy"]]
)
print("Best model by stratum:")
display(best_by_stratum)

final_parts = []
for _, row in best_by_stratum.iterrows():
    stratum = row["stratum"]
    model_name = row["supervised_model"]
    src_table = LGBM_TABLE if model_name == "lgbm" else XGB_TABLE
    part = con.execute(f"""
        SELECT *
        FROM {src_table}
        WHERE stratum = '{stratum}'
    """).df()
    final_parts.append(part)

# Carry forward dormant/unactivated/unknown from whichever model table exists first.
carry_table = LGBM_TABLE if table_exists(con, LGBM_TABLE) else XGB_TABLE
carry = con.execute(f"""
    SELECT *
    FROM {carry_table}
    WHERE LOWER(CAST(stratum AS VARCHAR)) IN ('dormant', 'unactivated', 'unknown')
""").df()
final_parts.append(carry)

final_df = pd.concat(final_parts, ignore_index=True)
final_df["assignment_source"] = np.where(
    final_df["stratum"].isin(MODEL_STRATA),
    "supervised_model_prediction",
    "carried_forward_v11_rule_or_pseudo_group",
)

save_df_to_table(con, final_df, FINAL_TABLE)
print(f"Saved final supervised cluster table: {FINAL_TABLE}")
print("Rows:", len(final_df))
display(final_df.groupby(["stratum", "supervised_model", "predicted_cluster_key"]).size().reset_index(name="n").head(50))


Best model by stratum:


,stratum,supervised_model,weighted_f1,balanced_accuracy
0,active,lgbm,0.998720,0.998422
4,at_risk,lgbm,0.996414,0.994331
2,cooling,lgbm,0.997402,0.997699


100% ▕██████████████████████████████████████▏ (00:00:03.12 elapsed)     
Saved final supervised cluster table: dev_supervised_cluster_membership_v1_final
Rows: 9381508


,stratum,supervised_model,predicted_cluster_key,n
0,active,lgbm,active_0,18015
1,active,lgbm,active_1,66682
2,active,lgbm,active_2,24658
3,active,lgbm,active_3,29409
4,active,lgbm,active_4,18549
5,active,lgbm,active_5,155034
6,active,lgbm,active_noise,105702
7,at_risk,lgbm,at_risk_0,436825
8,at_risk,lgbm,at_risk_1,145071
9,at_risk,lgbm,at_risk_2,210376


## 7. Build supervised cluster profile summary

In [8]:
summary_features = [c for c in ALL_FEATURES if c in profile_cols]
select_cols = [ID_COL] + summary_features
profile_df = con.execute(f"SELECT {', '.join(select_cols)} FROM {PROFILE_TABLE}").df()
membership_df = con.execute(f"SELECT * FROM {FINAL_TABLE}").df()
joined = membership_df.merge(profile_df, on=ID_COL, how="left")

agg = {
    ID_COL: "count",
    "prediction_confidence": "mean",
    "prediction_entropy": "mean",
}
for c in summary_features:
    agg[c] = "mean"

profile_summary = (
    joined
    .groupby(["stratum", "supervised_model", "predicted_cluster_key"], dropna=False)
    .agg(agg)
    .reset_index()
    .rename(columns={
        ID_COL: "n_developers",
        "prediction_confidence": "avg_prediction_confidence",
        "prediction_entropy": "avg_prediction_entropy",
    })
)
profile_summary["share_within_stratum"] = (
    profile_summary["n_developers"] /
    profile_summary.groupby("stratum")["n_developers"].transform("sum")
)
front = [
    "stratum", "supervised_model", "predicted_cluster_key",
    "n_developers", "share_within_stratum",
    "avg_prediction_confidence", "avg_prediction_entropy",
]
profile_summary = profile_summary[front + [c for c in profile_summary.columns if c not in front]]

save_df_to_table(con, profile_summary, PROFILE_SUMMARY_TABLE)
print(f"Saved profile summary table: {PROFILE_SUMMARY_TABLE}")
display(profile_summary.sort_values(["stratum", "n_developers"], ascending=[True, False]).head(40))


Saved profile summary table: dev_supervised_cluster_profile_summary_v1


,stratum,supervised_model,predicted_cluster_key,n_developers,share_within_stratum,avg_prediction_confidence,avg_prediction_entropy,active_non_builder_0_30d,activity_velocity_0_30_vs_30_90,avg_effort_rank_0_30d,avg_effort_rank_30_90d,avg_effort_rank_90_180d,build_share_lifetime,developer_effort_score,has_activity_0_30d,has_activity_30_90d,has_activity_90_180d,has_high_effort_0_30d,high_effort_share_lifetime,log_activity_count_0_30d,log_activity_count_30_90d,log_activity_count_90_180d,log_build_count_0_30d,log_build_count_30_90d,log_build_count_90_180d,log_clipped_lifetime_activity_count_p99,low_volume_builder_0_30d,persona_entropy,recent_build_flag,unique_activity_types_0_30d,unique_modalities_0_30d,weighted_recent_confidence_effort
5,active,lgbm,active_5,155034,0.370851,0.999937,1.590344e-04,1.000000,NaN,2.984810,0.000000,0.005789,0.000196,1.082724,1.0,0.000000,0.002645,0.984848,0.984015,0.693147,0.000000,0.002102,0.000000,0.000000,0.000018,0.704823,0.000000,0.003758,0.000000,1.000000,1.000000,1.348361
6,active,lgbm,active_noise,105702,0.252846,0.999673,8.544110e-04,0.744376,4.401090,1.267660,0.540179,0.360970,0.235229,2.031045,1.0,0.372358,0.227148,0.257706,0.241056,1.575473,0.780773,0.543741,0.454299,0.209764,0.204497,2.442167,0.116024,0.330746,0.255624,1.141350,1.103650,15.888645
1,active,lgbm,active_1,66682,0.159508,0.999959,9.570842e-05,1.000000,NaN,0.758223,0.000000,0.000480,0.000050,1.197769,1.0,0.000000,0.000720,0.999475,0.250227,1.999490,0.000000,0.000556,0.000000,0.000000,0.000000,1.995885,0.000000,0.024668,0.000000,2.000000,1.000000,1.375217
3,active,lgbm,active_3,29409,0.070348,0.999374,1.513954e-03,1.000000,1.041995,0.001326,0.541382,0.059833,0.000007,1.280912,1.0,1.000000,0.124350,0.000374,0.096480,2.126301,2.432114,0.369692,0.000000,0.000000,0.000000,3.004315,0.000000,0.000023,0.000000,1.000068,1.000000,0.611257
2,active,lgbm,active_2,24658,0.058984,0.999758,5.606134e-04,1.000000,NaN,2.546399,0.000000,0.004383,0.000537,1.911330,1.0,0.000000,0.003366,0.997769,0.563468,1.146747,0.000000,0.002647,0.000000,0.000000,0.000084,1.152698,0.000000,0.236728,0.000000,2.000000,2.000000,2.844089
4,active,lgbm,active_4,18549,0.044370,0.999952,1.008044e-04,1.000000,NaN,2.999030,0.000000,0.000000,0.000000,1.068473,1.0,0.000000,0.004475,0.999030,0.981940,0.693147,0.000000,0.003211,0.000000,0.000000,0.000000,0.707614,0.000000,0.427416,0.000000,1.000000,1.000000,1.349854
0,active,lgbm,active_0,18015,0.043093,0.999813,4.071534e-04,0.000000,0.916667,2.419248,0.000000,0.003363,0.498307,1.637898,1.0,0.000222,0.003608,0.997225,0.461656,1.174739,0.000260,0.003101,0.747661,0.000000,0.000638,1.183592,0.769137,0.291703,1.000000,2.061338,2.000056,2.802090
7,at_risk,lgbm,at_risk_0,436825,0.276318,0.998686,3.555193e-03,0.000000,NaN,0.000000,0.000000,2.151668,0.187602,0.821755,0.0,0.000000,1.000000,0.000000,0.461852,0.000000,0.000000,1.085320,0.000000,0.000000,0.322549,1.367429,0.000000,0.263964,0.000000,0.000000,0.000000,0.648384
12,at_risk,lgbm,at_risk_5,366741,0.231986,0.996563,1.011356e-02,0.000000,NaN,0.000000,0.000000,0.000000,0.325460,0.519924,0.0,0.000000,0.000000,0.000000,0.208884,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.004992,0.000000,0.355331,0.000000,0.000000,0.000000,0.000000
9,at_risk,lgbm,at_risk_2,210376,0.133076,0.999999,4.136130e-06,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.415676,0.0,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.098612,0.000000,0.210565,0.000000,0.000000,0.000000,0.000000


## 8. Optional parquet export

In [13]:
EXPORT_DIR = Path("toexport_clusters")
EXPORT_DIR.mkdir(exist_ok=True)

export_tables = [
    LGBM_TABLE,
    XGB_TABLE,
    FINAL_TABLE,
    RUN_STATS_TABLE,
    PROFILE_SUMMARY_TABLE,
]

existing_tables = set(con.execute("SHOW TABLES").df().iloc[:, 0].astype(str))
for table in export_tables:
    if table not in existing_tables:
        print("Skipping missing table:", table)
        continue
    out_path = EXPORT_DIR / f"{table}.parquet"
    con.execute(f"COPY {table} TO '{out_path.as_posix()}' (FORMAT PARQUET)")
    print("Exported:", out_path)


Exported: toexport_clusters/dev_supervised_cluster_membership_v1_lgbm.parquet
Exported: toexport_clusters/dev_supervised_cluster_membership_v1_xgb.parquet
Exported: toexport_clusters/dev_supervised_cluster_membership_v1_final.parquet
Exported: toexport_clusters/dev_supervised_cluster_run_stats_v1.parquet
Exported: toexport_clusters/dev_supervised_cluster_profile_summary_v1.parquet


## 9. Close connection

In [14]:
con.close()
print("DuckDB connection closed. Supervised cluster labeling notebook complete.")


DuckDB connection closed. Supervised cluster labeling notebook complete.
